In [ ]:
import csv
import io
import re
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds

from sympy import symbols, lambdify, sympify
from sympy.parsing.sympy_parser import parse_expr

from common_libs import check_vector


In [15]:
def show_iteration_table(result, decimals=8):
    """
    Hiển thị bảng:
        Iteration | x^k (vector) | Residual | Final

    Residual:
        r_k = ||x^k - x^(k-1)||_2
    """

    rows = []

    for k, x in sorted(result["res"].items()):

        vector_str = np.array2string(
            x,
            precision=decimals,
            separator=", ",
            suppress_small=False,
            threshold=np.inf,
            max_line_width=100000
        )

        rows.append({
            "Iteration (k)": k,
            "x^k (vector)": vector_str,
            "Residual ||x^k - x^(k-1)||₂":
                result["residual_history"].get(
                    k, np.nan
                ),
            "Final": k == result["iterations"]
        })

    table = pd.DataFrame(rows)

    table["Residual ||x^k - x^(k-1)||₂"] = (
        table["Residual ||x^k - x^(k-1)||₂"]
        .map(
            lambda v: f"{v:.3e}"
            if pd.notna(v) else "—"
        )
    )

    display(table)

    return table


In [16]:
rng = np.random.default_rng(2026)

T, n = 1000, 100

factors = np.column_stack([
    rng.normal(0.007, 0.035, size=T),
    rng.normal(0.000, 0.018, size=T)
])

loadings = np.vstack([
    rng.uniform(0.6, 1.4, size=n),
    rng.normal(0.0, 0.8, size=n)
])

alpha = rng.normal(0.001, 0.002, size=n)

R = (
    factors @ loadings
    + alpha
    + rng.normal(0, 0.025, size=(T, n))
)

print("R.shape =", R.shape)

R.shape = (1000, 100)


In [17]:
mu_hat = R.mean(axis=0)
rho = float(mu_hat.mean())
lambda_l1 = 0.01
b = rho * np.ones(T)

x0 = np.ones(n) / n

# Toán tử B
def B(x):
    return (2.0 / T) * R.T @ (R @ x - b)

# Ràng buộc cho phép chiếu
mu_scale = float(np.std(mu_hat))

A_eq = np.vstack([
    np.ones(n),
    (mu_hat - rho) / mu_scale
])

b_eq = np.array([1.0, 0.0])

constraint = LinearConstraint(A_eq, b_eq, b_eq)
bounds = Bounds(np.zeros(n), np.full(n, np.inf))

# Toán tử giải J_(gamma A) = P_C
def prox(z, gamma):
    z = np.asarray(z, dtype=float)

    result = minimize(
        fun=lambda x: 0.5 * np.linalg.norm(x - z)**2,
        x0=x0,
        jac=lambda x: x - z,
        method="SLSQP",
        bounds=bounds,
        constraints=[constraint],
        options={"ftol": 1e-12, "maxiter": 200}
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result.x

# Hằng số Lipschitz
L = (2.0 / T) * np.linalg.norm(R, ord=2)**2

# Cỡ bước chung cho FBM và MFBSM
gamma = 0.9 / L

# Giá trị hàm mục tiêu
def objective(x):
    return (
        np.linalg.norm(R @ x - b)**2 / T
        + lambda_l1 * np.linalg.norm(x, 1)
    )

print("L =", L)
print("gamma =", gamma)
print("rho =", rho)

L = 0.2905078592404491
gamma = 3.098022898083054
rho = 0.008886271241233326


# THUẬT TOÁN FORWARD-BACKWARD SPLITTING METHOD (FBM)

Điều kiện FBM:

- $A$ là toán tử đa trị đơn điệu cực đại (tồn tại resolvent)
- $B$ là toán tử đơn trị thường được giả thiết $\beta$ cocoercive
- $J_{\gamma A} = (I + \gamma A)^{-1}$ là toán tử giải của A
- $\gamma > 0$ là cỡ bước ($0<\gamma<2\beta$)

### Mã giả Forward–Backward Splitting Method (FBM)

**Bước 0 (Khởi tạo).**

Chọn điểm ban đầu $x^0\in\mathcal H$, cỡ bước $\gamma\in(0,2\beta)$, trong đó $B$ là toán tử $\beta$-cocoercive.

Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
y^k=x^k-\gamma B(x^k).
$$

**Bước 2.** Tính điểm lặp tiếp theo:

$$
x^{k+1}=J_{\gamma A}(y^k).
$$

**Bước 3.** Đặt $k:=k+1$ và quay lại Bước 1.

In [18]:
from FBM import fbm

# Ví dụ
a = np.array([3.0, 4.0])
x0 = np.array([100.0, -100.0])

# Toán tử B
def B(x):
    return x - a

# Phép chiếu lên quả cầu đóng
def proj_ball(x, center, radius):
    d = x - center
    nd = np.linalg.norm(d)

    if nd <= radius:
        return x.copy()

    return center + radius * d / nd

# Toán tử giải
def prox(y, gamma):
    return proj_ball(
        y,
        center=np.zeros(2),
        radius=1.0
    )

# Hằng số Lipschitz
L = 1.0
gamma = 1.0 / L

result = fbm(
    x0=x0,
    B=B,
    prox=prox,
    gamma=gamma,
    max_iter=10000,
    tol=1e-8,
    iter_list=[1, 10, 100, 1000, 10000]
)

table = show_iteration_table(result)

,Iteration (k),x^k (vector),Residual ||x^k - x^(k-1)||₂,Final
0,0,"[ 100., -100.]",—,False
1,1,"[0.6, 0.8]",1.416e+02,False
2,2,"[0.6, 0.8]",0.000e+00,True


# THUẬT TOÁN MODIFIED FORWARD–BACKWARD SPLITTING METHOD (MFBSM) CỦA TSENG

Điều kiện MFBSM:

- $A$ là toán tử đa trị đơn điệu cực đại (tồn tại resolvent).
- $B$ là toán tử đơn trị, đơn điệu và liên tục Lipschitz với hằng số $L>0$.
- Tập nghiệm $(A+B)^{-1}(0)$ khác rỗng.
- $J_{\alpha_k A}=(I+\alpha_k A)^{-1}$ là toán tử giải của $A$.
- $\alpha_k>0$ là cỡ bước, với dãy $\{\alpha_k\}$ thỏa mãn:

$$
\{\alpha_k\}\subset[a,b]\subset\left(0,\frac1L\right).
$$

Khác với FBM thông thường, MFBSM không yêu cầu toán tử $B$ phải cocoercive mà chỉ cần đơn điệu và liên tục Lipschitz.


### Mã giả Modified Forward–Backward Splitting Method (MFBSM_Tseng)

**Bước 0 (Khởi tạo).**

Chọn điểm ban đầu $x^0\in\mathcal H$, dãy cỡ bước $\{\alpha_k\}\subset[a,b]\subset(0,1/L)$, trong đó $B$ là toán tử đơn điệu và liên tục Lipschitz với hằng số $L>0$.

Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
z^k=x^k-\alpha_k B(x^k).
$$

**Bước 2.** Tính:

$$
y^k=J_{\alpha_k A}(z^k).
$$

**Bước 3.** Thực hiện bước hiệu chỉnh:

$$
x^{k+1}
=
y^k-\alpha_k\left(B(y^k)-B(x^k)\right).
$$

**Bước 4.** Đặt $k:=k+1$ và quay lại Bước 1.

In [19]:
from MFBSM_Tseng import mfbm_tseng

a = np.array([3.0, 4.0])
x0 = np.array([100.0, -100.0])

def B(x):
    return x - a

def proj_ball(x, center, radius):
    d = x - center
    nd = np.linalg.norm(d)

    if nd <= radius:
        return x.copy()

    return center + radius * d / nd

def prox(y, gamma):
    return proj_ball(
        y,
        center=np.zeros(2),
        radius=1.0
    )

L = 1.0
gamma = 0.9 / L

result = mfbm_tseng(
    x0=x0,
    B=B,
    prox=prox,
    gamma=gamma,
    max_iter=10000,
    tol=1e-8,
    iter_list=[1, 10, 100, 1000, 10000]
)

table = show_iteration_table(result)

,Iteration (k),x^k (vector),Residual ||x^k - x^(k-1)||₂,Final
0,0,"[ 100., -100.]",—,False
1,1,"[ 90.08930164, -90.0450024 ]",1.405e+01,False
2,10,"[ 35.49583139, -35.01672747]",5.458e+00,False
3,100,"[0.60335187, 0.79669394]",5.107e-04,False
4,206,"[0.60000006, 0.79999994]",9.275e-09,True



# THUẬT TOÁN RELAXED FORWARD–BACKWARD SPLITTING METHOD (RFBSM)

**Điều kiện RFBSM:**

- $A:\mathcal H\to 2^{\mathcal H}$ là toán tử đa trị đơn điệu cực đại.
- $B:\mathcal H\to\mathcal H$ là toán tử đơn trị, đơn điệu và liên tục Lipschitz với hằng số $L>0$.
- Tập nghiệm $\Omega=(A+B)^{-1}(0)\neq\varnothing$.
- $J_{\gamma_k A}=(I+\gamma_k A)^{-1}$ là toán tử giải của $A$.
- $\gamma_0>0$ là cỡ bước khởi tạo.
- $\{\theta_k\}\subset[a,b]\subset(0,1]$ là dãy tham số nới lỏng.
- $\{\mu_k\}\subset[c,d]\subset(0,1)$ là dãy tham số điều khiển cỡ bước.

### Mã giả Relaxed Forward–Backward Splitting Method (RFBSM)

**Bước 0 (Khởi tạo).**

Chọn điểm ban đầu $x^0\in\mathcal H$, cỡ bước $\gamma_0>0$, hai dãy tham số $\{\theta_k\}\subset[a,b]\subset(0,1]$ và $\{\mu_k\}\subset[c,d]\subset(0,1)$.

Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
y^k=J_{\gamma_k A}(x^k-\gamma_k B(x^k)).
$$

**Bước 2.** Tính điểm lặp tiếp theo:

$$
x^{k+1}=(1-\theta_k)x^k+\theta_k y^k
+\theta_k\gamma_k\big(B(x^k)-B(y^k)\big).
$$

**Bước 3.** Cập nhật cỡ bước:

$$
\gamma_{k+1}=
\begin{cases}
\displaystyle
\min\left\{\gamma_k,
\frac{\mu_k\|x^k-y^k\|}
{\|B(x^k)-B(y^k)\|}\right\},
& B(x^k)\neq B(y^k),\\[10pt]
\gamma_k,
& B(x^k)=B(y^k).
\end{cases}
$$

**Bước 4.** Đặt $k:=k+1$ và quay lại Bước 1.
